In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)
logging.getLogger('urllib3').setLevel(logging.ERROR)

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

E0000 00:00:1772978149.544476      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772978149.602403      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772978150.031790      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772978150.031824      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772978150.031826      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772978150.031829      55 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
# ================================================================
# 1. IMPORTS
# ================================================================
import pandas as pd
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from openai import OpenAI

In [3]:
# ================================================================
# 2. LLM CLIENT (ONLY for extracting keywords from user free text)
# ================================================================
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="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"
)

def extract_keywords_from_user_input(user_text):
    prompt = f"""You are a medical keyword extraction tool.

From the patient's description below, extract ONLY short medical keywords or phrases.

INCLUDE:
- Symptoms (e.g. fever, chest pain, dizziness, rash)
- Conditions (e.g. diabetes, high blood pressure)
- Risk factors (e.g. smoking, family history)
- Demographics (e.g. age, gender)
- Behaviours (e.g. heavy alcohol use)

EXCLUDE:
- Filler words and sentences
- Non-medical terms

FORMAT: Return ONLY keywords separated by ' | ' on a single line. Nothing else.

PATIENT TEXT: {user_text}"""

    completion = client.chat.completions.create(
        model="meta/llama-3.3-70b-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=512,
        stream=False
    )
    return completion.choices[0].message.content.strip()

In [4]:
# ================================================================
# 3. LOAD DATA
# ================================================================
df = pd.read_csv('/kaggle/input/datasets/karekezijoelnew01/dataset/data.csv')
print(f"Total rows loaded: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

Total rows loaded: 1169
Columns: ['disease_name', 'symptoms', 'risk_factors', 'specialist']


In [5]:
# ================================================================
# 4. FILL MISSING VALUES
# ================================================================
for col in ['disease_name', 'symptoms', 'risk_factors', 'specialist']:
    df[col] = df[col].fillna('').astype(str).str.strip()

print(f"Rows after fill: {len(df)}")

Rows after fill: 1169


In [6]:
# ================================================================
# 5. DISEASE LOOKUP TABLE
# ================================================================
disease_lookup = (
    df[['disease_name', 'symptoms', 'risk_factors', 'specialist']]
    .drop_duplicates(subset='disease_name')
    .set_index('disease_name')
)
print(f"Unique diseases in lookup: {len(disease_lookup)}")

Unique diseases in lookup: 1169


In [7]:
# ================================================================
# 6. BUILD VOCABULARY + WEIGHTED ENCODING
# ================================================================
clinicalbert = SentenceTransformer('emilyalsentzer/Bio_ClinicalBERT')

WEIGHT_SYMPTOMS = 1.0
WEIGHT_RISK_FACTORS = 0.5

all_keywords = set()
for col in ['symptoms', 'risk_factors']:
    for text in df[col]:
        for kw in text.split('|'):
            kw = kw.strip().lower()
            if kw:
                all_keywords.add(kw)

keyword_list = sorted(all_keywords)
keyword_to_idx = {k: i for i, k in enumerate(keyword_list)}
vocab_size = len(keyword_list)
print(f"Vocabulary size: {vocab_size}")

# Assign weight based on which column keyword appears in most
keyword_weight = {}
for kw in keyword_list:
    sym_count = sum(1 for text in df['symptoms'] if kw in [k.strip().lower() for k in text.split('|')])
    rf_count = sum(1 for text in df['risk_factors'] if kw in [k.strip().lower() for k in text.split('|')])
    keyword_weight[kw] = WEIGHT_SYMPTOMS if sym_count >= rf_count else WEIGHT_RISK_FACTORS

# Encode vocab with ClinicalBERT for fuzzy matching
print("Encoding vocabulary with ClinicalBERT...")
keyword_embeddings = clinicalbert.encode(keyword_list, show_progress_bar=True)


No sentence-transformers model found with name emilyalsentzer/Bio_ClinicalBERT. Creating a new one with mean pooling.


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Vocabulary size: 16131
Encoding vocabulary with ClinicalBERT...


Batches:   0%|          | 0/505 [00:00<?, ?it/s]

In [8]:
# ================================================================
# 7. SYMPTOM RARITY INDEX
# ================================================================
symptom_disease_count = {}
for kw in keyword_list:
    count = 0
    for text in df['symptoms']:
        if kw in [k.strip().lower() for k in text.split('|')]:
            count += 1
    symptom_disease_count[kw] = count

In [9]:
# ================================================================
# 8. ENCODING FUNCTION
# ================================================================
def text_to_multihot(keyword_string):
    vec = np.zeros(vocab_size, dtype=np.float32)
    if not keyword_string.strip():
        return vec
    keywords = [k.strip().lower() for k in keyword_string.split('|') if k.strip()]
    for kw in keywords:
        if kw in keyword_to_idx:
            idx = keyword_to_idx[kw]
            vec[idx] = keyword_weight.get(kw, 1.0)
        else:
            kw_emb = clinicalbert.encode([kw])
            sims = cosine_similarity(kw_emb, keyword_embeddings)[0]
            best_idx = np.argmax(sims)
            if sims[best_idx] > 0.7:
                vec[best_idx] = keyword_weight.get(keyword_list[best_idx], 1.0)
    return vec

# Encode training data
print("Encoding training data...")
X_list = []
for _, row in df.iterrows():
    vec = np.zeros(vocab_size, dtype=np.float32)
    for kw in row['symptoms'].split('|'):
        kw = kw.strip().lower()
        if kw and kw in keyword_to_idx:
            vec[keyword_to_idx[kw]] = WEIGHT_SYMPTOMS
    for kw in row['risk_factors'].split('|'):
        kw = kw.strip().lower()
        if kw and kw in keyword_to_idx:
            vec[keyword_to_idx[kw]] = WEIGHT_RISK_FACTORS
    X_list.append(vec)

X = np.vstack(X_list)
print(f"X shape: {X.shape}")

Encoding training data...
X shape: (1169, 16131)


In [10]:
# ================================================================
# 9. ENCODE LABELS
# ================================================================
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['disease_name'])
num_classes = len(label_encoder.classes_)
print(f"Classes: {num_classes}")

Classes: 1169


In [11]:
# ================================================================
# 10. BUILD MODEL
# ================================================================
model = Sequential([
    Input(shape=(vocab_size,)),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

I0000 00:00:1772978392.435724      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13215 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │     8,259,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1169)           │       150,801 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,577,681 (32.72 MB)

 Trainable params: 8,576,145 (32.72 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [12]:
# ================================================================
# 11. DATA AUGMENTATION + TRAIN
# ================================================================
def augment_data(X_orig, y_orig, augments_per_sample=30, min_keep=2):
    X_aug_list = [X_orig.copy()]
    y_aug_list = [y_orig.copy()]

    for i in range(len(X_orig)):
        vec = X_orig[i]
        on_indices = np.where(vec > 0)[0]
        if len(on_indices) < min_keep:
            continue
        for _ in range(augments_per_sample):
            n_keep = np.random.randint(min_keep, len(on_indices) + 1)
            selected = np.random.choice(on_indices, size=n_keep, replace=False)
            new_vec = np.zeros_like(vec)
            new_vec[selected] = vec[selected]
            X_aug_list.append(new_vec.reshape(1, -1))
            y_aug_list.append(np.array([y_orig[i]]))

    X_aug = np.vstack(X_aug_list)
    y_aug = np.concatenate(y_aug_list)
    return X_aug, y_aug

print("Augmenting with partial symptom subsets...")
X_aug, y_aug = augment_data(X, y, augments_per_sample=30, min_keep=2)
print(f"Original: {len(X)} samples -> Augmented: {len(X_aug)} samples")

shuffle_idx = np.random.permutation(len(X_aug))
X_aug = X_aug[shuffle_idx]
y_aug = y_aug[shuffle_idx]

history = model.fit(
    X_aug, y_aug,
    epochs=50,
    batch_size=64,
    verbose=1
)

Augmenting with partial symptom subsets...
Original: 1169 samples -> Augmented: 36239 samples
Epoch 1/50


I0000 00:00:1772978404.519765     157 service.cc:152] XLA service 0x7c1240005880 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772978404.519801     157 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1772978404.910480     157 cuda_dnn.cc:529] Loaded cuDNN version 91002


 30/567 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.0086 - loss: 7.0849

I0000 00:00:1772978407.412638     157 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


567/567 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.3767 - loss: 4.5263
Epoch 2/50
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9405 - loss: 0.3238
Epoch 3/50
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9635 - loss: 0.1724
Epoch 4/50
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9712 - loss: 0.1196
Epoch 5/50
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9748 - loss: 0.1045
Epoch 6/50
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9729 - loss: 0.1010
Epoch 7/50
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9765 - loss: 0.0928
Epoch 8/50
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9777 - loss: 0.0793
Epoch 9/50
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9786 - loss: 0.0754
Epoch 10/50
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9782 - loss: 0.0784
Epoch 11/50
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9769 - loss: 0.0811
Epoch 12/50
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accurac

In [13]:
# ================================================================
# 12. HELPER FUNCTIONS
# ================================================================
def predict_disease(keywords_text, top_k=3):
    multihot = text_to_multihot(keywords_text).reshape(1, -1)
    on_bits = np.where(multihot[0] > 0)[0]
    matched = [keyword_list[i] for i in on_bits]
    print(f"  [DEBUG] Matched {len(matched)} keywords: {matched}")

    probs = model.predict(multihot, verbose=0)[0]
    top_idx = np.argsort(probs)[::-1][:top_k]

    results = []
    for idx in top_idx:
        disease = label_encoder.inverse_transform([idx])[0]
        confidence = probs[idx]
        results.append({'disease': disease, 'confidence': confidence})
    return results


def get_disease_symptoms(disease_name):
    if disease_name not in disease_lookup.index:
        return set()
    raw = disease_lookup.loc[disease_name, 'symptoms']
    return {s.strip().lower() for s in raw.split('|') if s.strip()}


def get_specialist(disease_name):
    if disease_name not in disease_lookup.index:
        return "general practitioner"
    return disease_lookup.loc[disease_name, 'specialist']


def should_skip_followup(predictions, num_matched_keywords):
    if num_matched_keywords < 6:
        return False
    if len(predictions) < 2:
        return True
    if predictions[0]['confidence'] >= 0.75 and predictions[1]['confidence'] <= 0.15:
        return True
    return False


def get_top_symptoms_for_disease(disease_name, user_kws, shared_by_all, n=3):
    """
    Get the N rarest, most unique symptoms for a disease.
    Excludes symptoms the user already mentioned and shared by all top 3.
    """
    symptoms = get_disease_symptoms(disease_name)
    candidates = []

    for symptom in symptoms:
        if symptom in user_kws:
            continue
        if symptom in shared_by_all:
            continue

        total_diseases_with = symptom_disease_count.get(symptom, 999)
        rarity = 1.0 / max(total_diseases_with, 1)
        candidates.append({'symptom': symptom, 'rarity': rarity})

    candidates.sort(key=lambda x: x['rarity'], reverse=True)
    return [c['symptom'] for c in candidates[:n]]


def symptom_to_question(symptom):
    symptom = symptom.strip().lower()
    risk_phrases = ['family history', 'smoking', 'alcohol', 'drug use',
                    'tobacco', 'exposure', 'surgery', 'transplant', 'history']
    if any(rp in symptom for rp in risk_phrases):
        return f"Do you have {symptom}?"
    else:
        return f"Do you experience {symptom}?"


def ask_questions(symptoms_list, disease_name, question_number_start):
    """
    Ask yes/no questions for a list of symptoms.
    Returns (yes_keywords, yes_count, no_count, next_question_number).
    """
    yes_keywords = []
    yes_count = 0
    no_count = 0
    q_num = question_number_start

    for symptom in symptoms_list:
        question = symptom_to_question(symptom)
        print(f"\n  {q_num}. {question}")
        while True:
            answer = input("     Your answer (yes/no): ").strip().lower()
            if answer in ['yes', 'no', 'y', 'n']:
                break
            print("     Please type Yes or No.")

        if answer in ['yes', 'y']:
            yes_keywords.append(symptom)
            yes_count += 1
        else:
            no_count += 1
        q_num += 1

    return yes_keywords, yes_count, no_count, q_num

In [14]:
# ================================================================
# 13. FULL CONSULTATION
# ================================================================
QUESTIONS_PER_DISEASE = 3

def is_similar_symptom(symptom, asked_set):
    """Check if symptom is too similar to one already asked."""
    s = symptom.lower().strip()
    for asked in asked_set:
        a = asked.lower().strip()
        # One contains the other
        if s in a or a in s:
            return True
        # Check without plural
        if s.rstrip('s') == a.rstrip('s'):
            return True
        # Check word overlap - if 80%+ words match, too similar
        s_words = set(s.split())
        a_words = set(a.split())
        if len(s_words) > 0 and len(a_words) > 0:
            overlap = len(s_words & a_words) / min(len(s_words), len(a_words))
            if overlap >= 0.8:
                return True
    return False


def run_consultation():
    print("=" * 60)
    print("  CURAMEDICA - MEDICAL SYMPTOM CHECKER")
    print("=" * 60)

    user_text = input("\nDescribe your symptoms: ").strip()
    while not user_text:
        print("  Please describe your symptoms.")
        user_text = input("Describe your symptoms: ").strip()

    print("\nAnalyzing your symptoms...")
    keywords = extract_keywords_from_user_input(user_text)
    print(f"Identified: {keywords}")

    initial_predictions = predict_disease(keywords, top_k=3)

    print("\n" + "=" * 60)
    print("  INITIAL ASSESSMENT (Top 3)")
    print("=" * 60)
    for i, pred in enumerate(initial_predictions, 1):
        print(f"  {i}. {pred['disease']} ({pred['confidence']:.1%})")

    matched_count = int(np.sum(text_to_multihot(keywords) > 0))

    if should_skip_followup(initial_predictions, matched_count):
        print("\n  Model is confident. Skipping follow-up questions.")
        final_disease = initial_predictions[0]
    else:
        user_kws = {k.strip().lower() for k in keywords.split('|') if k.strip()}
        all_sets = [get_disease_symptoms(p['disease']) for p in initial_predictions]
        if len(all_sets) >= 3:
            shared_by_all = all_sets[0] & all_sets[1] & all_sets[2]
        elif len(all_sets) == 2:
            shared_by_all = all_sets[0] & all_sets[1]
        else:
            shared_by_all = set()

        print(f"\n  Follow-up questions. Answer Yes or No.")
        print("-" * 60)

        all_confirmed = []
        disease_scores = {}
        all_asked_symptoms = set()
        q_num = 1
        winner = None

        for pred in initial_predictions:
            disease_name = pred['disease']
            init_conf = pred['confidence']

            top_symptoms = get_top_symptoms_for_disease(
                disease_name, user_kws, shared_by_all, n=QUESTIONS_PER_DISEASE + 3
            )

            # Filter out symptoms similar to already asked ones
            filtered = []
            for s in top_symptoms:
                if not is_similar_symptom(s, all_asked_symptoms):
                    filtered.append(s)
                if len(filtered) >= QUESTIONS_PER_DISEASE:
                    break

            top_symptoms = filtered

            if not top_symptoms:
                disease_scores[disease_name] = {'yes': 0, 'no': 0, 'conf': init_conf}
                continue

            print(f"\n  Checking for: {disease_name}")
            yes_kws, yes_count, no_count, q_num = ask_questions(
                top_symptoms, disease_name, q_num
            )

            all_confirmed.extend(yes_kws)
            user_kws.update(k.lower() for k in yes_kws)
            all_asked_symptoms.update(s.lower() for s in top_symptoms)

            disease_scores[disease_name] = {
                'yes': yes_count,
                'no': no_count,
                'conf': init_conf
            }

            if yes_count >= 2:
                print(f"\n  Strong match found.")
                winner = disease_name
                break

            if yes_count == 0:
                print(f"  Ruled out: {disease_name}")

        if winner:
            enriched = keywords + ' | ' + ' | '.join(all_confirmed)
            final_preds = predict_disease(enriched, top_k=3)
            final_disease = None
            for p in final_preds:
                if p['disease'] == winner:
                    final_disease = p
                    break
            if not final_disease:
                final_disease = {'disease': winner, 'confidence': 0.90}
        else:
            if all_confirmed:
                enriched = keywords + ' | ' + ' | '.join(all_confirmed)
                final_preds = predict_disease(enriched, top_k=3)
                final_disease = final_preds[0]
            else:
                best = None
                best_score = -1
                for name, scores in disease_scores.items():
                    score = scores['yes'] - scores['no'] + scores['conf']
                    if score > best_score:
                        best_score = score
                        best = name
                if best:
                    final_disease = {'disease': best, 'confidence': 0.50}
                else:
                    final_disease = initial_predictions[0]

    specialist = get_specialist(final_disease['disease'])

    print("\n" + "=" * 60)
    print("  FINAL ASSESSMENT")
    print("=" * 60)
    print(f"  {final_disease['disease']} -- {final_disease['confidence']:.2%} confidence")
    print(f"\n  Recommended specialist: {specialist}")
    print("\n  NOTE: This is not a medical diagnosis.")
    print("  Please consult a qualified healthcare professional.")
    print("=" * 60)


run_consultation()

  CURAMEDICA - MEDICAL SYMPTOM CHECKER



Describe your symptoms:  i have a fever i cough a lot i shiver i have a headache



Analyzing your symptoms...
Identified: fever | cough | shiver | headache
  [DEBUG] Matched 4 keywords: ['cough', 'fever', 'headache', 'jolt']

  INITIAL ASSESSMENT (Top 3)
  1. typhoid fever (64.8%)
  2. syphilis (11.7%)
  3. legionnaires' disease (10.1%)

  Follow-up questions. Answer Yes or No.
------------------------------------------------------------

  Checking for: typhoid fever

  1. Do you experience trouble passing stool?


     Your answer (yes/no):  no



  2. Do you experience swollen belly?


...


  8. Do you experience confusion?


     Your answer (yes/no):  no



  9. Do you experience diarrhea?


     Your answer (yes/no):  no


  [DEBUG] Matched 5 keywords: ['chills', 'cough', 'fever', 'headache', 'jolt']

  FINAL ASSESSMENT
  typhoid fever -- 99.97% confidence

  Recommended specialist: infectious disease specialist

  NOTE: This is not a medical diagnosis.
  Please consult a qualified healthcare professional.


In [15]:
# ================================================================
# 14. SAVE MODEL (uncomment when ready)
# ================================================================
SAVE_DIR = '/kaggle/working//model'
import os
os.makedirs(SAVE_DIR, exist_ok=True)

model.save(os.path.join(SAVE_DIR, 'disease_classifier_v2.keras'))

with open(os.path.join(SAVE_DIR, 'label_encoder_v2.pkl'), 'wb') as f:
    pickle.dump(label_encoder, f)

np.save(os.path.join(SAVE_DIR, 'keyword_embeddings_v2.npy'), keyword_embeddings)

with open(os.path.join(SAVE_DIR, 'vocab_v2.pkl'), 'wb') as f:
    pickle.dump({
        'keyword_list': keyword_list,
        'keyword_to_idx': keyword_to_idx,
        'keyword_weight': keyword_weight,
        'symptom_disease_count': symptom_disease_count,
    }, f)

with open(os.path.join(SAVE_DIR, 'disease_lookup_v2.pkl'), 'wb') as f:
    pickle.dump(disease_lookup, f)

print("All files saved.")

All files saved.
